# Chat avec le Qwen3-14B obfusqué — accès via Tailscale

Le modèle obfusqué `qwen3-14b-h128-a1-h02` est servi **serverless sur Modal**
(A100-40GB). Un **proxy OpenAI-compatible** tourne sur `sanroque` (service
systemd `obfuscator-proxy`, clé de permutation locale), exposé sur le
**tailnet** par `tailscale serve` :

```
https://sanroque.tailc69141.ts.net:9443/v1   ← endpoint OpenAI (tailnet only)
```

Ce notebook fonctionne **depuis n'importe quelle machine du tailnet** (buho,
un autre poste…) — il suffit d'avoir Tailscale connecté. Le trafic est
chiffré par le tailnet ; la clé Π ne quitte jamais sanroque.

> Prérequis réseau : `tailscale up` sur la machine qui exécute ce notebook.

In [1]:
import json
import requests

# Endpoint OpenAI-compatible du proxy (via Tailscale, HTTPS magique)
BASE_URL = "https://sanroque.tailc69141.ts.net:9443/v1"
MODEL = "qwen3-14b-h128-a1-h02"

# 1. Vérifier la connexion (health + modèles disponibles)
r = requests.get(BASE_URL.replace("/v1", "") + "/health", timeout=30)
print("health:", r.json())
r = requests.get(BASE_URL + "/models", timeout=30)
print("modèles:", [m["id"] for m in r.json()["data"]])

health: {'status': 'ok', 'proxy': True}
modèles: ['qwen3-14b-h128-a1-h02']


## Envoyer un prompt

La fonction `chat()` envoie des messages au format OpenAI
(`/v1/chat/completions`) et renvoie la réponse du modèle.

In [2]:
def chat(messages, max_tokens=200, temperature=None):
    """Appelle le modèle obfusqué via le proxy Tailscale (format OpenAI)."""
    payload = {"model": MODEL, "messages": messages, "max_tokens": max_tokens}
    if temperature is not None:
        payload["temperature"] = temperature   # accepté mais ignoré (greedy)
    r = requests.post(BASE_URL + "/chat/completions",
                      json=payload, timeout=600)
    r.raise_for_status()
    d = r.json()
    return (d["choices"][0]["message"]["content"],
            d["usage"]["prompt_tokens"], d["usage"]["completion_tokens"])

# --- EXEMPLE 1 : question simple ---
reponse, pt, ct = chat([
    {"role": "user", "content": "Quelle est la capitale de la France ? "
                                "Réponds en un mot."},
], max_tokens=50)
print("Q : Quelle est la capitale de la France ?")
print("R :", reponse)
print(f"   ({pt} tokens prompt, {ct} tokens réponse)")

Q : Quelle est la capitale de la France ?
R : Paris.
   (29 tokens prompt, 3 tokens réponse)


## Exemples d'usage

- **Résumé** : le modèle résume un texte en français (format wiki possible).
- **Multi-tour** : l'historique est renvoyé à chaque appel.

In [3]:
# --- EXEMPLE 2 : résumé court ---
texte = ("La confidentialité des échanges avec un LLM hébergé dans le cloud "
         "est un enjeu majeur pour les professions réglementées. Une approche "
         "consiste à transformer à la fois les données et le modèle, de sorte "
         "que le serveur ne manipule que des jetons permutés, illisibles sans "
         "la clé de permutation détenue par le client.")
reponse, _, _ = chat([
    {"role": "user",
     "content": f"Résume ce texte en 2 phrases : {texte}"},
], max_tokens=120)
print("Résumé :", reponse)

Résumé : La confidentialité des communications avec un LLM hébergé en cloud est cruciale pour les métiers réglementés. Une solution consiste à permuter les données et le modèle afin que seuls les clients puissent décrypter les informations grâce à une clé de permutation.


In [4]:
# --- EXEMPLE 3 : conversation multi-tour ---
messages = [
    {"role": "user", "content": "Quelle est la capitale de la Belgique ?"},
    {"role": "assistant", "content": "Bruxelles."},
    {"role": "user", "content": "Et la capitale de la Suisse ? Réponds en un mot."},
]
reponse, _, _ = chat(messages, max_tokens=50)
print("R :", reponse)

R : Berne.


## Chat interactif (à exécuter manuellement)

Exécutez la cellule ci-dessous pour une petite boucle de conversation dans le
notebook. Tapez `quit` pour sortir.

> Ne pas inclure cette cellule dans une exécution automatique (`nbconvert`)
> : elle attend une saisie clavier.

In [ ]:
def chat_interactif():
    messages = []
    print("Chat avec", MODEL, "(tape 'quit' pour sortir)\n")
    while True:
        user = input("Vous : ")
        if user.strip().lower() in ("quit", "exit"):
            break
        messages.append({"role": "user", "content": user})
        reponse, _, _ = chat(messages, max_tokens=250)
        print("Modèle :", reponse)
        messages.append({"role": "assistant", "content": reponse})

# chat_interactif()   # ← décommentez pour lancer (interactif)

Chat avec qwen3-14b-h128-a1-h02 (tape 'quit' pour sortir)

Vous : Qui était Henri de Navarre ?
Modèle : Henri de Navarre, également connu sous le nom d'**Henri IV**, était un roi important de France, qui a régné du **13 mai 1589** jusqu'à sa mort en **1598**. Il est l'un des rois les plus célèbres de la monarchie française, notamment pour avoir mis fin aux guerres de Religion et pour avoir été le premier roi catholique de France après une longue période de conflits entre catholiques et protestants (les Huguenots).

### Points clés sur Henri IV :

- **Origine** : Il était le fils d’Antoine de Bourbon, comte de Béarn, et d’Isabelle de Clèves. Il était donc originaire de Navarre, ce qui lui donnait un titre royal étranger avant qu’il ne devienne roi de France.
  
- **Conversion religieuse** : Il était initialement protestant (huguenot), mais il se convertit au catholicisme en **1593**, affirmant célèbrement : **"Paris vaut bien une messe"**. Cette conversion
Vous : Jeanne a trois soeurs e

## Appel d'outils (function calling)

Le modèle obfusqué supporte l'appel d'outils au format OpenAI : on fournit
des définitions de fonctions (`tools`), le modèle répond par un
`tool_calls` (nom + arguments JSON) s'il a besoin d'un outil, le client
l'exécute et renvoie le résultat (`role: "tool"`), puis le modèle donne sa
réponse finale.

> Le proxy applique le format Qwen3 (`<tool_call>…</tool_call>`) et traduit
> vers le format OpenAI — rien de tout cela n'est visible côté Modal.

In [ ]:
# Outil de démonstration : météo (simulée localement)
OUTILS_METEO = [{"type": "function", "function": {
    "name": "get_weather",
    "description": "Donne la météo d'une ville",
    "parameters": {"type": "object",
                   "properties": {"city": {"type": "string"}},
                   "required": ["city"]}}}]

def executer_meteo(arguments):
    """Simule l'exécution de l'outil météo (à remplacer par un vrai appel)."""
    ville = arguments.get("city", "inconnue")
    return f"Il fait 18°C et ensoleillé à {ville}."

def chat_tools(messages, tools=None):
    """Comme chat(), mais avec les définitions d'outils."""
    payload = {"model": MODEL, "messages": messages, "max_tokens": 200}
    if tools:
        payload["tools"] = tools
    r = requests.post(BASE_URL + "/chat/completions",
                      json=payload, timeout=600)
    r.raise_for_status()
    return r.json()["choices"][0]

### Tour 1 — le modèle demande l'outil

La question exige une information que le modèle ne connaît pas (la météo) :
il répond par un `tool_calls` au lieu d'un texte.

In [ ]:
messages = [{"role": "user",
              "content": "Quel temps fait-il à Paris ?"}]
r1 = chat_tools(messages, OUTILS_METEO)
print("finish_reason :", r1["finish_reason"])
for tc in r1["message"]["tool_calls"]:
    print("outil appelé   :", tc["function"]["name"])
    print("arguments      :", tc["function"]["arguments"])

### Tour 2 — exécution de l'outil et réponse finale

On exécute l'outil localement, on ajoute le résultat au fil de discussion
(`role: "tool"`), et le modèle formule la réponse pour l'utilisateur.

In [ ]:
# exécuter l'outil localement
tc = r1["message"]["tool_calls"][0]
arguments = json.loads(tc["function"]["arguments"])
resultat = executer_meteo(arguments)
print("résultat outil :", resultat)

# renvoyer le résultat au modèle (tour 2)
messages.append({"role": "assistant", "content": None,
                 "tool_calls": [tc]})
messages.append({"role": "tool", "tool_call_id": tc["id"],
                 "content": resultat})
r2 = chat_tools(messages, OUTILS_METEO)
print("réponse finale :", r2["message"]["content"])

### Boucle agent générique (optionnel)

Une boucle qui laisse le modèle appeler plusieurs outils à la suite
jusqu'à une réponse textuelle finale.

In [ ]:
def agent(messages, tools, executeur, max_tours=4):
    """Boucle outil → exécution → renvoi, jusqu'à une réponse texte."""
    for _ in range(max_tours):
        r = chat_tools(messages, tools)
        if r["finish_reason"] != "tool_calls":
            return r["message"]["content"]
        msgs_assistant = {"role": "assistant", "content": None,
                          "tool_calls": r["message"]["tool_calls"]}
        messages.append(msgs_assistant)
        for tc in r["message"]["tool_calls"]:
            nom, args = tc["function"]["name"], json.loads(
                tc["function"]["arguments"])
            print(f"  → outil {nom}({args})")
            resultat = executeur[nom](args)
            messages.append({"role": "tool", "tool_call_id": tc["id"],
                             "content": resultat})
    return "(trop de tours)"

# exécuteur d'outils (à étendre selon vos besoins)
executeur = {"get_weather": executer_meteo}
print(agent([{"role": "user",
              "content": "Quel temps fait-il à Paris et à Bruxelles ?"}],
            OUTILS_METEO, executeur))

## Notes

- **Modèle** : Qwen3-14B obfusqué (h>0, α_e=1,0/α_h=0,2) — défense mesurée :
  VMA gate 10,5 % / W_e·W_h 0 % / combiné 7,05 % ; qualité Q&A −1,1 % vs base.
- **Sécurité** : la clé de permutation reste sur sanroque (proxy local) ; le
  tailnet chiffre le transport ; Modal ne voit que des ids permutés.
- **Coût** : serverless A100-40GB, facturé au temps GPU réellement allumé
  (scale-to-zero) + cold start après inactivité (~1-2 min).
- **Limites** : pas de streaming (réponse complète), greedy (temperature
  ignorée), le proxy est mono-utilisateur sur le tailnet.
- Gestion du service sur sanroque :
  `systemctl --user status obfuscator-proxy` (logs : `journalctl --user -u
  obfuscator-proxy -f`).